# Colab

In [1]:
import os
from google.colab import drive
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/CS-4452 Computer Vision - Code')

Mounted at /content/drive


# Import

In [18]:
from dotenv import load_dotenv
import os
from PIL import Image
import numpy as np
from tqdm import tqdm
import copy
from scipy.spatial.distance import directed_hausdorff
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

# Paths

In [3]:
load_dotenv()
path = os.getenv("DATA_PATH")

In [4]:
train_path = os.path.join(path, "train")
train_img_path = os.path.join(train_path, "images")
train_mask_path = os.path.join(train_path, "masks")

In [5]:
val_path = os.path.join(path, "validation")
val_img_path = os.path.join(val_path, "images")
val_mask_path = os.path.join(val_path, "masks")

In [6]:
model_path = "./model"

# Data

In [7]:
def load_images(dir):
    images = []
    filenames = sorted(os.listdir(dir))

    for name in tqdm(filenames, desc="Loading Images"):
        path = os.path.join(dir, name)
        img = Image.open(path)
        img = np.array(img, dtype=np.float32) / 255.0
        img = np.expand_dims(img, axis=0)
        images.append(img)

    images = np.array(images)
    return torch.tensor(images)

In [8]:
def load_masks(dir):
    masks = []
    filenames = sorted(os.listdir(dir))

    for name in tqdm(filenames, desc="Loading Masks"):
        path = os.path.join(dir, name)
        mask = Image.open(path)
        mask = np.array(mask, dtype=np.float32)
        mask = (mask > 0).astype(np.float32)
        mask = np.expand_dims(mask, axis=0)
        masks.append(mask)

    masks = np.array(masks)
    return torch.tensor(masks)

In [9]:
train_img_data = load_images(train_img_path)
train_mask_data = load_masks(train_mask_path)
val_img_data = load_images(val_img_path)
val_mask_data = load_masks(val_mask_path)
print("\n")
print(train_img_data.shape)
print(train_mask_data.shape)
print(val_img_data.shape)
print(val_mask_data.shape)
print(train_img_data.min(), train_img_data.max())
print(train_mask_data.unique())

Loading Masks: 100%|██████████| 672/672 [00:09<00:00, 70.53it/s] 




torch.Size([3580, 1, 128, 128])
torch.Size([3580, 1, 128, 128])
torch.Size([672, 1, 128, 128])
torch.Size([672, 1, 128, 128])
tensor(0.) tensor(1.)
tensor([0., 1.])


# Model

In [10]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.conv(x)

In [11]:
class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super(UNet, self).__init__()
        self.down1 = DoubleConv(in_channels, 64)
        self.down2 = DoubleConv(64, 128)
        self.down3 = DoubleConv(128, 256)
        self.down4 = DoubleConv(256, 512)

        self.up1 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.conv1 = DoubleConv(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.conv2 = DoubleConv(256, 128)
        self.up3 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.conv3 = DoubleConv(128, 64)

        self.outc = nn.Conv2d(64, out_channels, 1)

    def forward(self, x):
        x1 = self.down1(x)
        x2 = self.down2(F.max_pool2d(x1, 2))
        x3 = self.down3(F.max_pool2d(x2, 2))
        x4 = self.down4(F.max_pool2d(x3, 2))

        x = self.up1(x4)
        x = torch.cat([x, x3], dim=1)
        x = self.conv1(x)

        x = self.up2(x)
        x = torch.cat([x, x2], dim=1)
        x = self.conv2(x)

        x = self.up3(x)
        x = torch.cat([x, x1], dim=1)
        x = self.conv3(x)

        logits = self.outc(x)
        return logits


# Loss Function

In [12]:
class BCEDiceLoss(nn.Module):
    def __init__(self, bce_weight=0.5):
        super(BCEDiceLoss, self).__init__()
        self.bce_weight = bce_weight
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, inputs, targets, smooth=1e-5):
        bce_loss = self.bce(inputs, targets)
        inputs = torch.sigmoid(inputs)
        inputs = inputs.view(-1)
        targets = targets.view(-1)
        intersection = (inputs * targets).sum()
        dice_loss = 1 - (2. * intersection + smooth) / (inputs.sum() + targets.sum() + smooth)

        return self.bce_weight * bce_loss + (1 - self.bce_weight) * dice_loss

# Train

In [13]:
def calculate_metrics(preds, targets):
    preds = torch.sigmoid(preds) > 0.5
    targets = targets > 0.5

    # Move to CPU for metrics
    preds = preds.cpu().numpy()
    targets = targets.cpu().numpy()

    batch_size = preds.shape[0]
    acc_sum = 0
    dice_sum = 0
    iou_sum = 0
    hd_sum = 0

    for i in range(batch_size):
        p = preds[i, 0]
        t = targets[i, 0]

        # Accuracy
        acc = (p == t).mean()
        acc_sum += acc

        # Dice and IoU
        intersection = np.logical_and(p, t).sum()
        union = np.logical_or(p, t).sum()

        if t.sum() == 0 and p.sum() == 0:
            dice = 1.0
            iou = 1.0
        else:
            dice = 2. * intersection / (p.sum() + t.sum() + 1e-6)
            iou = intersection / (union + 1e-6)

        dice_sum += dice
        iou_sum += iou

        # Hausdorff Distance
        p_coords = np.argwhere(p)
        t_coords = np.argwhere(t)
        if len(p_coords) == 0 and len(t_coords) == 0:
            hd = 0.0
        elif len(p_coords) == 0 or len(t_coords) == 0:
            hd = np.sqrt(p.shape[0]**2 + p.shape[1]**2)
        else:
            hd = max(directed_hausdorff(p_coords, t_coords)[0],
                     directed_hausdorff(t_coords, p_coords)[0])
        hd_sum += hd

    return acc_sum, dice_sum, iou_sum, hd_sum

In [14]:
def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs, patience, model_save_path):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    os.makedirs(model_save_path, exist_ok=True)
    best_model_wts = copy.deepcopy(model.state_dict())
    best_loss = float('inf')
    epochs_no_improve = 0

    # Track losses and metrics
    history = {
        'train_loss': [],
        'val_loss': [],
        'val_accuracy': [],
        'val_dice': [],
        'val_iou': [],
        'val_hausdorff': []
    }

    for epoch in range(num_epochs):
        print(f"Epoch {epoch+1}/{num_epochs}")
        print("-" * 10)

        # -------- TRAIN PHASE --------
        model.train()
        running_loss = 0.0

        train_loop = tqdm(train_loader, desc='Training', leave=False)
        for inputs, labels in train_loop:
            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            with torch.set_grad_enabled(True):
                outputs = model(inputs)
                loss = criterion(outputs, labels)

                loss.backward()
                optimizer.step()

            running_loss += loss.item() * inputs.size(0)

        epoch_train_loss = running_loss / len(train_loader.dataset)
        history['train_loss'].append(epoch_train_loss)

        # -------- VALIDATION PHASE --------
        model.eval()
        running_val_loss = 0.0

        run_val_acc = 0.0
        run_val_dice = 0.0
        run_val_iou = 0.0
        run_val_hd = 0.0

        val_loop = tqdm(val_loader, desc='Validation', leave=False)
        for inputs, labels in val_loop:
            inputs = inputs.to(device)
            labels = labels.to(device)

            with torch.set_grad_enabled(False):
                outputs = model(inputs)
                loss = criterion(outputs, labels)

            running_val_loss += loss.item() * inputs.size(0)

            # Calculate metrics
            a, d, iou, hd = calculate_metrics(outputs, labels)
            run_val_acc += a
            run_val_dice += d
            run_val_iou += iou
            run_val_hd += hd

        dataset_len = len(val_loader.dataset)
        epoch_val_loss = running_val_loss / dataset_len
        epoch_val_acc = run_val_acc / dataset_len
        epoch_val_dice = run_val_dice / dataset_len
        epoch_val_iou = run_val_iou / dataset_len
        epoch_val_hd = run_val_hd / dataset_len

        history['val_loss'].append(epoch_val_loss)
        history['val_accuracy'].append(epoch_val_acc)
        history['val_dice'].append(epoch_val_dice)
        history['val_iou'].append(epoch_val_iou)
        history['val_hausdorff'].append(epoch_val_hd)

        print(f"Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f}")
        print(f"Val Acc: {epoch_val_acc:.4f} | Val Dice: {epoch_val_dice:.4f} | Val IoU: {epoch_val_iou:.4f} | Val Hausdorff: {epoch_val_hd:.4f}")

        # -------- EARLY STOPPING --------
        if epoch_val_loss < best_loss:
            best_loss = epoch_val_loss
            best_model_wts = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0

            save_dest = os.path.join(model_save_path, "best_unet.pth")
            torch.save(model.state_dict(), save_dest)
            print(f"Best model saved to {save_dest}")
        else:
            epochs_no_improve += 1
            print(f"Early stop counter: {epochs_no_improve} / {patience}")

        if epochs_no_improve >= patience:
            print("Early stopping triggered due to no improvement.")
            break

    print(f"Training complete. Best Val Loss: {best_loss:.4f}")
    model.load_state_dict(best_model_wts)
    return model, history

In [15]:
# Hyperparameters
BATCH_SIZE = 16
LEARNING_RATE = 0.0001
EPOCHS = 100
PATIENCE = 10
BCE_WEIGHT = 0.5

train_dataset = TensorDataset(train_img_data, train_mask_data)
val_dataset = TensorDataset(val_img_data, val_mask_data)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [17]:
model = UNet(in_channels=1, out_channels=1).to(device)

criterion = BCEDiceLoss(bce_weight=BCE_WEIGHT)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

trained_model, loss_history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    num_epochs=EPOCHS,
    patience=PATIENCE,
    model_save_path=model_path
)

Epoch 1/100
----------


Train Loss: 0.6304 | Val Loss: 0.5837
Val Acc: 0.9649 | Val Dice: 0.4077 | Val IoU: 0.3676 | Val Hausdorff: 77.8834
Best model saved to ./model/best_unet.pth
Epoch 2/100
----------


Train Loss: 0.5587 | Val Loss: 0.5333
Val Acc: 0.9873 | Val Dice: 0.6675 | Val IoU: 0.6382 | Val Hausdorff: 43.2525
Best model saved to ./model/best_unet.pth
Epoch 3/100
----------


Train Loss: 0.5139 | Val Loss: 0.4865
Val Acc: 0.9843 | Val Dice: 0.6500 | Val IoU: 0.6124 | Val Hausdorff: 47.0069
Best model saved to ./model/best_unet.pth
Epoch 4/100
----------


Train Loss: 0.4722 | Val Loss: 0.4616
Val Acc: 0.9869 | Val Dice: 0.6834 | Val IoU: 0.6484 | Val Hausdorff: 40.9259
Best model saved to ./model/best_unet.pth
Epoch 5/100
----------


...

Train Loss: 0.2410 | Val Loss: 0.3469
Val Acc: 0.9908 | Val Dice: 0.7190 | Val IoU: 0.6885 | Val Hausdorff: 38.2132
Early stop counter: 6 / 10
Epoch 24/100
----------


Train Loss: 0.2380 | Val Loss: 0.3613
Val Acc: 0.9904 | Val Dice: 0.7082 | Val IoU: 0.6759 | Val Hausdorff: 37.3615
Early stop counter: 7 / 10
Epoch 25/100
----------


Train Loss: 0.2285 | Val Loss: 0.3583
Val Acc: 0.9900 | Val Dice: 0.7039 | Val IoU: 0.6757 | Val Hausdorff: 40.8633
Early stop counter: 8 / 10
Epoch 26/100
----------


Train Loss: 0.2272 | Val Loss: 0.3713
Val Acc: 0.9901 | Val Dice: 0.7150 | Val IoU: 0.6850 | Val Hausdorff: 37.1935
Early stop counter: 9 / 10
Epoch 27/100
----------


Train Loss: 0.2260 | Val Loss: 0.3984
Val Acc: 0.9899 | Val Dice: 0.6885 | Val IoU: 0.6623 | Val Hausdorff: 42.7033
Early stop counter: 10 / 10
Early stopping triggered due to no improvement.
Training complete. Best Val Loss: 0.3403


In [20]:
with open("history.json", "w") as f:
    json.dump(loss_history, f)